# Simulation core — one sepsis model, every engine: composed, ensembled, orchestrated, benchmarked

_Investigation `biogears-sbml-sepsis` — coder reproduction notebook._

**Question.** How can a compositional simulation core support interchangeable physiological
models, systematic perturbation studies, scalable execution, mechanistic
comparison, dynamic intervention, and evidence-based scientific validation? Taking
one real model, the BioGears sepsis core, each is a runnable artifact:
  01 INTERCHANGEABILITY — which model properties are preserved when the
     implementation changes (BioGears C++, extracted SBML/Tellurium, surrogate)?
  02 ENSEMBLES — how do severity and physiological parameters drive progression
     toward shock?
  03 ORCHESTRATION — can registered modules form a reproducible investigation with
     no new integration code, including inside a Nextflow pipeline?
  04 SCALABLE EXECUTION — where does acceleration preserve useful predictions, and
     where does it fail? Includes the Rust viva-core backend.
  05 MECHANISTIC FIDELITY — what explains the inflammatory agreement and the
     hemodynamic divergence between engines?
  06 DYNAMIC COUPLING — can coupled physiology and treatment produce consistent
     intervention responses (untreated vs early vs delayed)?
  07 EVIDENCE & ACCEPTANCE — can the Workbench turn runs into traceable evidence
     with explicit pass / fail / pending acceptance criteria?
The showcase is about what the core lets scientists test, not only what software it
can execute — while staying honest about where the engines agree and where they do not.

A guided tour of the simulation core's capabilities on one real model. The same
BioGears sepsis core is run under interchangeable engines (BioGears C++, two exact
SBML integrators — Tellurium and COPASI — a PyTorch surrogate, and a Rust execution
backend) and used to
demonstrate, each as a runnable artifact: COMPOSITION (study 01 — the same spec
under three engines by one node swap), ENSEMBLES (study 02 — a severity × engine
batch in one sweep), ORCHESTRATION/WORKFLOWS (study 03 — the four-verb API composes
modules into a pipeline and hands an Antimony/SBML file between engines, and a
Nextflow pipeline fans the real composite across a severity grid), PERFORMANCE &
EXECUTION BACKENDS (study 04 — where the compute goes, a ~40,000× surrogate tier,
and the same composite run under the Rust viva-core backend), and CO-SIMULATION
(study 06 — two processes exchanging state every tick: the physiology engine
coupled to a closed-loop antibiotic controller, with untreated/early/delayed arms),
and EVIDENCE & ACCEPTANCE (study 07 — turning runs into traceable pass/fail/pending
acceptance criteria). A fidelity backbone (study 05) keeps it honest: where the
engines agree, and where they do not.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/inflect-sim-core/inflect-sim-core').is_dir():
    REPO = Path('/home/runner/work/inflect-sim-core/inflect-sim-core')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_simulation_core.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: `00-simulation-core`

_study.yaml not found — skipped._


## Study: `01-composition`

_study.yaml not found — skipped._


## Study: `02-ensembles`

_study.yaml not found — skipped._


## Study: `03-workflows`

_study.yaml not found — skipped._


## Study: `04-performance`

_study.yaml not found — skipped._


## Study: `05-fidelity`

_study.yaml not found — skipped._


## Study: `06-cosimulation`

_study.yaml not found — skipped._


## Study: `07-evidence`

_study.yaml not found — skipped._


## Study: `08-reduced-model`

_study.yaml not found — skipped._


## Study: `09-event-architecture`

_study.yaml not found — skipped._


## Open decisions
- Follow-up (05, staged): test one candidate mechanism for the hemodynamic divergence (e.g. a simplified vasodilatory effect) with a controlled perturbation; quantify whether it reduces the time-to-shock discrepancy. Do not present documented omissions as proven causes.
- Follow-up (02, staged): a 5×5 severity × physiology-parameter sweep with time-to-hypotension, minimum MAP, and tissue integrity — an actual biological response surface (compute-heavy on the stiff engine).
- Follow-up (01, partial): a port / units / initialization / ownership compatibility report across the three implementations.
- Follow-up (03, partial): automated shock-event detection and metric extraction as a reusable workflow step.
- Done this pass (06): untreated vs early vs delayed treatment counterfactuals + a coupling-step convergence test.
- Done this pass (04): surrogate admissibility on a withheld severity (screening-grade vs mechanistic).
- Done this pass (07): pass / fail / pending acceptance gates linked to evidence artifacts.
